## Main training loop

In [1]:
# from alpha_worms.reflex_endpoint.game_engine import GameEngine
# from alpha_worms.nn import WormsPredictor
# from alpha_worms.main import Worms

# # 1. Initialize Game and Brain
# game = GameEngine() 
# predictor = WormsPredictor()

# # 2. Define Hyperparameters
# args = {
#     'lr': 0.001,
#     'numMCTSSims': 100,
#     'cpuct': 1.41,
#     'num_iterations': 5,
#     'num_episodes': 10,
#     'batch_size': 64,
#     'max_buffer_size': 20000,
#     'checkpoint': './checkpoints/'
# }

# # 3. Initialize the Trainer
# worms_trainer = Worms(game, predictor, args)

# # 4. START TRAINING
# # This will run for a while, saving a .pth.tar file after every iteration
# worms_trainer.learn()

## Model move prediction

In [ ]:
from alpha_worms.reflex_endpoint.game_engine import GameEngine
from alpha_worms.nn import WormsPredictor
import numpy as np

# 1. Initialize
game = GameEngine()
predictor = WormsPredictor()

# 2. Load the brain you just trained
predictor.load_checkpoint(folder='./checkpoints/', filename='checkpoint_15.pth.tar')

# 3. Get a starting board
initial_state = game.get_initial_state()

# 4. Ask the brain what it thinks! (Get RAW probabilities)
raw_policy, value = predictor.predict(initial_state)

# ---------------------------------------------------------
# 5. MASK AND RE-NORMALIZE (Same logic as MCTS Expansion)
# ---------------------------------------------------------
valid_moves = game.get_allowed_moves(initial_state)

# Multiply raw probabilities by the valid moves mask (illegal moves become 0.0)
masked_policy = raw_policy * valid_moves
sum_probs = np.sum(masked_policy)

# Re-normalize so the remaining valid probabilities sum to 1.0
if sum_probs > 0:
    policy = masked_policy / sum_probs
else:
    # Fallback in case the untrained network output 0.0 for all valid moves
    policy = valid_moves / np.sum(valid_moves)
# ---------------------------------------------------------

print(f"Value of the starting board: {value:.3f} (closer to 1 means Player 1 is favored)")
print(f"Top 3 move probabilities:")

# Grab the indices of the highest probabilities from the MASKED policy
top_moves = np.argsort(policy)[::-1][:3]

for move in top_moves:
    # Print legal moves
    print(f"Legal moves mask: {valid_moves}")
    # print(initial_state) # Uncomment if you want to see the DTO every loop
    
    # We now print from the cleaned 'policy', not 'raw_policy'
    print(f"Move ID {move}: {policy[move] * 100:.1f}%")
    print(f"Full Cleaned Policy:\n{np.round(policy, 3)}")
    print('-----------------------------')

Model loaded from ./checkpoints/checkpoint_15.pth.tar
Value of the starting board: -0.026 (closer to 1 means Player 1 is favored)
Top 3 move probabilities:
Legal moves mask: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0]
Move ID 22: 15.3%
Full Cleaned Policy:
[0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.134 0.118 0.126 0.123 0.104 0.124 0.153 0.118
 0.   ]
-----------------------------
Legal moves mask: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0]
Move ID 16: 13.4%
Full Cleaned Policy:
[0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.134 0.118 0.126 0.123 0.104 0.124 0.153 0.118
 0.   ]
-----------------------------
Legal moves mask: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0]
Move ID 18: 12.6%
Full Cleaned Policy:
[0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    